# 02 — Cross-Sectional Momentum Research (Example)

**Research question**: Can cross-sectional momentum generate stable
out-of-sample risk-adjusted returns across liquid multi-asset ETFs after
transaction costs and volatility targeting?

**Universe**: SPY, QQQ, IWM, EFA, EEM, TLT, GLD, VNQ — US large/small
cap, developed/emerging ex-US equities, long treasuries, gold, real estate.

**Signal**: 12-month return excluding the most recent month (standard
momentum construction to avoid short-term reversal), ranked cross-sectionally
each month; long the top half.

**Hypothesis** (H1): after realistic transaction costs and out-of-sample
validation, the strategy delivers a positive risk-adjusted return (Sharpe > 0)
that is reasonably stable across sub-periods.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from quantlab.config import ExperimentConfig
from quantlab.data.loader import DataLoader
from quantlab.backtesting.runner import run_backtest_from_config

config = ExperimentConfig.from_yaml("../configs/momentum_sp500.yaml")
data, report = DataLoader().load(config)
len(data), report.row_count


## Run the backtest

In [ ]:
result = run_backtest_from_config(data, config, data_quality_report=report)
print(result.summary())


## Equity curve vs benchmark

In [ ]:
equity = result.equity_curve
benchmark_returns = result.benchmark_returns

if benchmark_returns is None:
    raise RuntimeError("No benchmark returns are available for this backtest.")

bench_equity = float(equity.iloc[0]) * (1 + benchmark_returns).cumprod()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(equity.index, equity.to_numpy(dtype=float), label="Strategy", lw=1.5)
ax.plot(bench_equity.index, bench_equity.to_numpy(dtype=float), label=f"Benchmark ({config.benchmark_label})", lw=1.2, ls="--")
ax.set_title("Cross-sectional momentum vs benchmark")
ax.legend()
plt.show()


## Performance by year

In [ ]:
from quantlab.reporting.tables import subperiod_table
table = subperiod_table(result)
table


## Out-of-sample validation (walk-forward)

Parameters are selected on a validation block and only then evaluated on an
untouched test block — the test period is never used to choose
`lookback_period`, `skip_period` or `top_fraction`. The notebook calls the same
configured-or-default grid helper as the CLI, so both interfaces evaluate
identical candidates.


In [ ]:
from quantlab.validation import WalkForwardValidator, parameter_grid_for_config

train_window = config.validation.train_window or 500
validation_window = config.validation.validation_window or 126
test_window = config.validation.test_window or 126

validator = WalkForwardValidator(config)
wf = validator.run(
    data,
    parameter_grid=parameter_grid_for_config(config),
    train_window=train_window,
    validation_window=validation_window,
    test_window=test_window,
    expanding=config.validation.expanding,
)
print(f"{len(wf.folds)} walk-forward folds")
wf.summary_table()


In [ ]:
oos_metrics = wf.oos_metrics(config.periods_per_year, config.risk_free_rate)
print(f"Out-of-sample Sharpe: {oos_metrics.get('sharpe_ratio', 0):.2f}")
print(f"Out-of-sample CAGR:   {oos_metrics.get('cagr', 0):.2%}")
print(f"Parameter stability (coefficient of variation): {wf.parameter_stability()}")


In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wf.oos_equity.index, wf.oos_equity.to_numpy(dtype=float), color="#2563eb", lw=1.3)
ax.set_title("Stitched out-of-sample equity curve (walk-forward test blocks only)")
plt.show()


## Conclusion

The strategy's full-sample and walk-forward out-of-sample Sharpe ratios are
relatively close, which provides some reassurance against severe overfitting.
However, the observed risk-adjusted performance remains modest and does not
demonstrate a persistent ability to outperform the market. These results are
historical, depend on the stated assumptions and execution costs, and do not
guarantee future performance.
